# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-11_dataset_and_frozen.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day3/T1-11_dataset_and_frozen.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [1]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day3"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day3" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


COLAB TRACK
 course: G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course
 cwd   : G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day3
 Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.


# T1-11 Dataset + what is frozen

**Time:** 09:00–10:30 (architecture briefing 09:00–09:15) &nbsp;|&nbsp; **Evidence:** dataset report + canary list in `logs/`

**Notebook learning outcomes**

- Shade the architecture card: **frozen** base vs a small **adapter**.
- Split **burst** training from **standing** inference (N11 / N7).
- Inspect an SFT JSONL: instruction, input, output, **train vs held-out**, **canaries**.

You do **not** train today. There is no GPU job in this notebook.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — two pictures before any rows

Day 1 architecture card (keep it):

```
input text → tokenizer → embedding → transformer blocks → LM head → next token
```

**Training** updates weights. **Inference** uses frozen weights. Yesterday RAG did **not** update weights. It copied a paragraph into the prompt.

**Fine-tuning** (the thing people say when they mean “make it ours”) usually does **not** rewrite all of OLMo. A **LoRA adapter** is a small extra file sitting on some of the blocks. The base stays frozen. You can disable the file. That is rollback.

![Frozen base vs a small adapter](../../tier1/day3/images/t3_frozen_vs_adapter.jpg)

**Burst vs standing** is the cost card from T1-03, said as architecture:

| | Burst | Standing |
|---|---|---|
| What | A training (or LoRA) **job** | The chat **API** that stays up |
| When | Hours, then stop the extra GPU | All day on `:8000` / `:8001` |
| Who asked | CPO “train on judgments” | CPO / TTDF / Finance “own the door” |
| Fail | Buying one box “for AI” with no split | Assuming a LoRA replaces the API |

![Burst training vs standing inference](../../tier1/day3/images/t3_burst_vs_standing.jpg)

**Extra words:**

- **SFT** — supervised fine-tune: each row is instruction + input + desired output.
- **JSONL** — one JSON object per line. That is `data/sft_rows.jsonl`.
- **Held-out** — rows you do **not** train on, so you can see whether the phrasing generalises.
- **Canary** — planted PII. Must not be in a real train split. If the adapter later quotes it, the set was dirty.
- **Licence inheritance** — the adapter is licensed as the **base** plus **every row**. T1-12b.


## Ministry link

| EOI type | What to look at in the rows |
|---|---|
| **Labour / Health** | FT-01, FT-02, FT-07: C2 phrasing, quote, do not hire |
| **all PII papers** | FT-08, FT-09 canaries (Maya Reid, burgess) |
| **Trade** | FT-04: do not reset a password |
| **Defence OPR** | FT-06: do not award |
| **Defence assets** | FT-10: you are not the register |
| **CPO / TTDF / Finance** | Burst vs standing on the card — not a judgment dump in this file |

This lab is **N1, N5, N12** (open-weight base; adapter is a derivative).


## 1. List the rows


In [2]:
# [RUN]
import civicdesk_ft as ft

print("CivicDesk SFT rows (fiction). Do not replace with ministry judgments.\n")
ft.print_rows()


CivicDesk SFT rows (fiction). Do not replace with ministry judgments.

id      split    canary instruction
FT-01   train    False  Screen this CivicDesk application against C2. Do not hire.
FT-02   train    False  Screen this CivicDesk application against C2. Do not hire.
FT-03   train    False  Check this notice against N1. Do not publish.
FT-04   train    False  ICT ticket. Do not reset a password.
FT-05   train    False  Cite CivicDesk attachment overnight rule or stop.
FT-06   train    False  OPR question. Do not award.
FT-07   held-out False  Screen this CivicDesk application against C2. Do not hire.
FT-08   train    True   Summarise this application.
FT-09   train    True   Extract the household.
FT-10   train    False  Register question. You are not the register.


### Teaching note

`split: train` is what a real job would see. `split: held-out` (FT-07) is a C2 variant you keep back. `canary: true` rows are **poison** if they enter training — they exist so you practise spotting them.


## 2. Dataset report — what stays frozen


In [3]:
# [RUN]
import json
import civicdesk_ft as ft

report = ft.dataset_report()
print()
print(json.dumps(report, indent=2))
ft.save_log("T1-11_report", report)


rows: 10
splits: {'train': 9, 'held-out': 1}
canaries (must stay out of a real train set): 2
  FT-08  all — N5  Applicant Maya Reid, National ID 999-00-1111, seeks the records post.
  FT-09  RDLG  Burgess form: 14 Harbour Close, income listed, names of minors.
base weights today: FROZEN. We do not update OLMo. An adapter would be a small extra file.

{
  "n": 10,
  "canaries": 2,
  "splits": {
    "train": 9,
    "held-out": 1
  },
  "frozen": true
}
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day3\logs\T1-11_report.json


### After you run

You should see canaries **FT-08** and **FT-09** named in the printout. `frozen: true` is the classroom rule: OLMo on disk is the same file as Day 1.

**Write-up:** which row would you **delete** before a real job, and which row is a fair C2 example?


## 3. Open one criteria row

Labour/Health shape: match text to C2, quote, do not hire.


In [4]:
# [CHANGE] try FT-02 (meets C2) or FT-07 (held-out).
ROW_ID = "FT-01"

import json
import civicdesk_ft as ft

row = {r["id"]: r for r in ft.load_rows()}[ROW_ID]
print(json.dumps(row, indent=2))


{
  "id": "FT-01",
  "split": "train",
  "canary": false,
  "instruction": "Screen this CivicDesk application against C2. Do not hire.",
  "input": "I completed a two-year diploma in library studies in 2019.",
  "output": "Human review required. Quoted span: 'diploma in library studies'. C2 is records management or equivalent; library studies is not automatic equivalent. Do not hire.",
  "stands_in_for": "Labour"
}


Look at `output`. It is a **script**, not a hire. The adapter, if it learned this, would only be copying a voice. The vacancy handbook can still change — that text still wants **RAG** (Day 2). Fine-tune does not replace HB-VAC-C2.


In [5]:
# [OPTIONAL] token-ish length of outputs — a stand-in for 'will this fit'
import civicdesk_ft as ft

for r in ft.load_rows():
    n = len(r["output"].split())
    print(f"{r['id']}  {n:3} words  canary={r['canary']}")


FT-01   24 words  canary=False
FT-02   21 words  canary=False
FT-03   22 words  canary=False
FT-04   14 words  canary=False
FT-05   13 words  canary=False
FT-06   16 words  canary=False
FT-07   15 words  canary=False
FT-08   12 words  canary=True
FT-09   13 words  canary=True
FT-10   14 words  canary=False
